In [2]:
import boto3
import json
from dotenv import load_dotenv
import os
from transformers import  AutoTokenizer
import pandas as pd
import random

from botocore.exceptions import ClientError

In [3]:
load_dotenv()

True

In [4]:
AWS_KEY = os.getenv("AWS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")

In [5]:
bedrock = boto3.client(service_name='bedrock', 
region_name='us-west-2', 
aws_access_key_id=AWS_KEY, 
aws_secret_access_key=AWS_SECRET_KEY)

In [6]:
response = bedrock.list_foundation_models(byProvider="meta")

for summary in response["modelSummaries"]:
    print(summary["modelId"])

meta.llama2-13b-chat-v1:0:4k
meta.llama2-13b-chat-v1
meta.llama2-70b-chat-v1:0:4k
meta.llama2-70b-chat-v1
meta.llama2-13b-v1:0:4k
meta.llama2-13b-v1
meta.llama2-70b-v1:0:4k
meta.llama2-70b-v1
meta.llama3-8b-instruct-v1:0
meta.llama3-70b-instruct-v1:0
meta.llama3-1-8b-instruct-v1:0
meta.llama3-1-70b-instruct-v1:0
meta.llama3-1-405b-instruct-v1:0


# Test Model Response

In [7]:
model_id = 'meta.llama3-1-405b-instruct-v1:0'
hf_model_name = "meta-llama/Meta-Llama-3.1-405B-Instruct"

In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(hf_model_name, padding_side="left")
tokenizer.pad_token = tokenizer.bos_token

In [9]:
temperature = 0.8
top_p=0.9
max_token_to_generate = 2000

In [10]:
def load_questions_to_df(question_file: str):
    """Load questions from a file into a DataFrame."""
    questions = []
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                questions.append(json.loads(line))
    
    df = pd.DataFrame([{
        "question_id": question["question_id"],
        "content": question["turns"][0]["content"],
        "cluster": question["cluster"]
    } for question in questions])
    
    return df

In [11]:
def prepare_prompts(df):
    """Prepare prompts dynamically based on question DataFrame."""
    prompts = []

    for _, row in df.iterrows():
        system_prompt = f"""
        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "{row['cluster']}".\n
        """

        user_message = f"{row['content']}"
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ]

        prompts.append(messages)

    df['prompt'] = prompts
    return df

In [12]:
# Load questions into a DataFrame
question_df = load_questions_to_df("arena-hard-auto/data/arena-hard-v0.1/question.jsonl")

# Prepare prompts
question_df = prepare_prompts(question_df)

# Select a random row from the DataFrame
example_question_df = question_df.sample(n=1, random_state=random.randint(0, len(question_df) - 1))

example_question_df

,question_id,content,cluster,prompt
197,742071e7f5c348e79834951803b5cd69,Please write GLSL code (both vertex shader and...,HLSL Shader Decompilation Refactoring,"[{'role': 'system', 'content': ' You a..."


In [13]:
message = example_question_df['prompt'].values[0]

prompt = tokenizer.apply_chat_template(
    message, 
    tokenize=False, 
    add_generation_prompt=True
)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "HLSL Shader Decompilation Refactoring".<|eot_id|><|start_header_id|>user<|end_header_id|>

Please write GLSL code (both vertex shader and fragment shader) for old-school raycasting.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [14]:
body = json.dumps({
    "prompt": prompt,
    "max_gen_len":max_token_to_generate,
    "temperature":temperature,
    "top_p":top_p
})

In [15]:
bedrock_runtime = boto3.client(service_name='bedrock-runtime', 
                            region_name='us-west-2', 
                            aws_access_key_id=AWS_KEY, 
                            aws_secret_access_key=AWS_SECRET_KEY)

In [16]:
response = bedrock_runtime.invoke_model(body=body, modelId=model_id, accept="application/json", contentType="application/json")
print(response)

{'ResponseMetadata': {'RequestId': 'ed8727d0-117c-4d26-8369-3d08c62a3a6c', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 10 Aug 2024 18:01:37 GMT', 'content-type': 'application/json', 'content-length': '2927', 'connection': 'keep-alive', 'x-amzn-requestid': 'ed8727d0-117c-4d26-8369-3d08c62a3a6c', 'x-amzn-bedrock-invocation-latency': '44328', 'x-amzn-bedrock-output-token-count': '683', 'x-amzn-bedrock-input-token-count': '93'}, 'RetryAttempts': 0}, 'contentType': 'application/json', 'body': <botocore.response.StreamingBody object at 0x131df7f70>}


In [17]:
# Colors to test the model
class bcolors:
    OKGREEN = '\033[92m'
    CBLUE   = '\33[34m'
    CVIOLET = '\33[35m'
    ENDC = '\033[0m'

In [18]:
def print_prompt(prompt, response, with_system=False):
    print("="*30 + f" Chat with  --- {model_id} ---  LLM using AWS Model API " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ {model_id} ]" + bcolors.ENDC)
    print(response["generation"])

In [19]:
model_response = json.loads(response["body"].read())

In [20]:
print_prompt(message, model_response, with_system=True)

============================== Chat with  --- meta.llama3-1-405b-instruct-v1:0 ---  LLM using AWS Model API ==============================

[ SYSTEM ]

        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "HLSL Shader Decompilation Refactoring".

        

[ USER ]
Please write GLSL code (both vertex shader and fragment shader) for old-school raycasting.

[ meta.llama3-1-405b-instruct-v1:0 ]
Here's a simple implementation of old-school raycasting in GLSL, using a vertex shader and a fragment shader. This code assumes you have a 2D texture representing the map, where each pixel's color corresponds to a wall type (or empty space).

**Vertex Shader:**
```glsl
#version 330 core

layout (location = 0) in vec3 aPos;

uniform mat4 projection;
uniform mat4 modelview;

void main()
{
    gl_Position = projection * modelview * vec4(aPos, 1.0);
}
```

**Fragment Shader:**
```gls

# Test Response Stream

In [82]:
def print_stream(prompt, streaming_response, with_system=False):
    print("="*30 + f" Chat with  --- {model_id} ---  LLM using vLLM " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ ASSISTANT ]" + bcolors.ENDC + "\n")
    for event in streaming_response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if "generation" in chunk:
            print(chunk["generation"], end="")

In [83]:
# Select a random row from the DataFrame
example_question_df = question_df.sample(n=1, random_state=random.randint(0, len(question_df) - 1))

message = example_question_df['prompt'].values[0]

prompt = tokenizer.apply_chat_template(
    message, 
    tokenize=False, 
    add_generation_prompt=True
)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "Calculating Pi in Python".<|eot_id|><|start_header_id|>user<|end_header_id|>

How to write a program in Python to calculate flight path<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [84]:
# Format the request payload using the model's native structure.
native_request = {
    "prompt": prompt,
    "max_gen_len": 2048,
    "temperature": 0.8,
    "top_p": 0.8
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    streaming_response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=model_id, body=request
    )
    print_stream(message, streaming_response, with_system=True)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

============================== Chat with  --- meta.llama3-1-405b-instruct-v1:0 ---  LLM using vLLM ==============================

[ SYSTEM ]

        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "Calculating Pi in Python".

        

[ USER ]
How to write a program in Python to calculate flight path 

[ ASSISTANT ]

To calculate the flight path of an object, we'll need to consider factors like initial velocity, angle of projection, and acceleration due to gravity. Here's a Python program to calculate the flight path using the equations of motion.

```python
import math
import matplotlib.pyplot as plt

def calculate_flight_path(initial_velocity, angle_of_projection):
    """
    Calculates the flight path of an object given the initial velocity and angle of projection.

    Args:
        initial_velocity (float): The initial velocity of the object in m/s.
        ang

In [ ]:
def chat_completion_awsbedrock(model, messages, temperature, max_tokens, api_dict=None, api_info=None):
    from transformers import  AutoTokenizer
    import boto3
    from botocore.exceptions import ClientError

    if 'api_secret_key' in api_dict:
        aws_key = api_dict["api_key"]
        aws_secret_key = api_dict["api_secret_key"]
    else:
        aws_key = os.environ["AWS_KEY"]
        aws_secret_key = os.environ["AWS_SECRET_KEY"]

    if 'aws_region' in api_dict:
        aws_region = api_dict["aws_region"]
    else:
        aws_region = 'us-west-2'

    # initialize the bedrock runtime client
    bedrock_runtime = boto3.client(service_name='bedrock-runtime', 
                            region_name=aws_region, 
                            aws_access_key_id=aws_key, 
                            aws_secret_access_key=aws_secret_key)


    # llama models in AWS Bedrock don't use a json-like chat template
    if 'model_type' in api_info:
        model_type = api_info["model_type"]
    else: 
        model_type = 'llama-3.1'
    
    # Llama Models Require the Prompt in one String
    if 'llama' in model_type:
        if model_type == 'llama-3.1':
            tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-405B-Instruct", padding_side="left")
        elif model_type == 'llama-3':
            tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-70B-Instruct", padding_side="left")
        else:
            tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-70B-Instruct", padding_side="left")

        tokenizer.pad_token = tokenizer.bos_token
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        body = json.dumps({
            "prompt": prompt,
            "max_gen_len":max_tokens,
            "temperature":temperature,
            "top_p":0.8
        })

    # Claude Models Require the Anthropic Version
    elif 'claude' in model_type:
        if 'anthropic_version' in api_info:
            anthropic_version = api_info["anthropic_version"]
        else:
            anthropic_version = 'bedrock-2023-05-31'

        body = json.dumps({
            "anthropic_version": anthropic_version,
            "max_tokens": max_tokens,
            "temperature":temperature,
            "messages": messages
        })
    
    # For a few of the other models there might be a different format as well. This needs to be checked
    else: 
        body = json.dumps({
            "max_tokens": max_tokens,
            "temperature":temperature,
            "messages": messages
        })

    try:
        response_json = bedrock_runtime.invoke_model(body=body, 
                                            modelId=model, 
                                            accept="application/json", 
                                            contentType="application/json")

        response = json.loads(response_json['Body'].read())
        output = response["generation"]

    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke '{model}'. Reason: {e}")
        output = API_ERROR_OUTPUT

    return output